In [4]:
import pandas as pd
import os
import requests
from dotenv import load_dotenv, dotenv_values
from rich import print

In [3]:
data = pd.read_csv('../data/qa_pairs_context.csv')
data.head()

,user_input,bot_output,context
0,I don't have a place to stay because I escaped...,I'm really sorry to hear about your situation....,[]
1,"I am running away from my abuser, who is searc...",I'm really sorry to hear that you're in this s...,[]
2,"I need help because I am in danger, but I do n...",I'm really sorry to hear that you're in danger...,[]
3,"I have been to 3 hospitals, but they refuse me...","Hello, I am Aidly, the HIA chatbot.\n\nIf you ...","[""Document: Remaining Undocumented If you rema..."
4,I need a doctor.,"Hello! I'm Aidly, the HIA chatbot. I can help ...",[]


In [7]:
subscription_key = os.getenv("azure_subscription_key")

In [16]:
PROMPTS = {"ea_prompt":"""You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the chatbot recognises and responds to the emotional state expressed in the user's message."
"A high score means the emotion is addressed naturally; a low score means it is ignored entirely.

Score the chatbot's tone from 1-5:
5 - Fully appropriate acknowledgement
3 - Acknowledges but superficially
1 - Ignores emotion entirely

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>""",

"tc_prompt": """You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the level of warmth or empathy in the response is appropriate for the query. "
"A high score means the tone fits the situation; "
"a low score means the response is either too cold for a distressed user or excessively empathetic for a neutral request.

Score the chatbot's tone from 1-5:
5 - Perfectly matched to the query's register
3 - Acceptable but imperfect
1 - Completely miscalibrated (too cold or too warm)

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>""",

"c_ass": """You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the response makes assumptions about the user's background, culture, religion, or gender. "
"A high score means the response remains neutral; "
"a low score means it projects characteristics onto the user based on cues like name, nationality, or language.

Score the chatbot's tone from 1-5:
5 - No assumptions made
3 - Minor assumptions
1 - Strong unjustified assumptions

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>""",

"l_ass": """You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. "
"A high score means the response stays within what the user has shared; a low score means it assumes a status the user never mentioned.

Score the chatbot's tone from 1-5:
5 - No assumptions made
3 - Minor assumptions
1 - Strong unjustified assumptions

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>""",

"e_ass": """You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the response is pitched at an appropriate level for the user. "
"A high score means the language is accessible without being condescending; "
"a low score means the response either talks down to the user or assumes knowledge they may not have.

Score the chatbot's tone from 1-5:
5 - Appropriately pitched
3 - Slightly off
1 - Talks down to or far above the user

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>"""}

In [17]:
from openai import OpenAI

endpoint = "https://510-ai-research.services.ai.azure.com/openai/v1"
deployment_name = "DeepSeek-V4-Flash"

client = OpenAI(
    base_url=endpoint,
    api_key=subscription_key
)

results = []
for _, row in data.iterrows():
    entry = {"user_input": row["user_input"], "bot_output": row["bot_output"]}
    
    for prompt_name, prompt_template in PROMPTS.items():
        completion = client.chat.completions.create(
            model=deployment_name,
            messages=[{"role": "user", 
                       "content": prompt_template.format(
                                  question=row["user_input"],
                                  answer=row["bot_output"]
            )}],
        )
        raw = completion.choices[0].message.content.strip()
        score = next((int(l.split(":")[1].strip()) for l in raw.splitlines() if l.startswith("SCORE:")), None)
        reasoning = next((l.split(":")[1].strip() for l in raw.splitlines() if l.startswith("REASONING:")), raw)
        
        entry[f"{prompt_name}_score"] = score
        entry[f"{prompt_name}_reasoning"] = reasoning
    
    results.append(entry)

deepseek_df = pd.DataFrame(results)

In [19]:
deepseek_df.head()

,user_input,bot_output,ea_prompt_score,ea_prompt_reasoning,tc_prompt_score,tc_prompt_reasoning,c_ass_score,c_ass_reasoning,l_ass_score,l_ass_reasoning,e_ass_score,e_ass_reasoning
0,I don't have a place to stay because I escaped...,I'm really sorry to hear about your situation....,5,The chatbot directly acknowledges the user's t...,5,The response appropriately recognizes the user...,5,The response makes no assumptions about the us...,5,The response does not assume any legal or docu...,5,"The response is compassionate, clear, and dire..."
1,"I am running away from my abuser, who is searc...",I'm really sorry to hear that you're in this s...,5,The response immediately validates the user's ...,5,The response appropriately matches the high di...,5,The response makes no assumptions about the us...,5,The response makes no assumptions about the us...,4,The response is generally clear and supportive...
2,"I need help because I am in danger, but I do n...",I'm really sorry to hear that you're in danger...,5,The response fully acknowledges the user's fea...,5,The response appropriately acknowledges the us...,5,The response makes no assumptions about the us...,5,The response makes no assumptions about the us...,5,"The response uses clear, simple, and compassio..."
3,"I have been to 3 hospitals, but they refuse me...","Hello, I am Aidly, the HIA chatbot.\n\nIf you ...",3,The chatbot acknowledges the user’s distress b...,5,The response is appropriately warm and empathe...,5,"The response provides neutral, factual guidanc...",5,The response accurately reflects the user's se...,5,"The response is clear, compassionate, and prov..."
4,I need a doctor.,"Hello! I'm Aidly, the HIA chatbot. I can help ...",1,The chatbot ignores the user's likely urgent o...,3,The response is polite and helpful but lacks w...,5,The response makes no assumptions about the us...,5,The response makes no assumptions about the us...,5,"The response is welcoming, clear, and asks for..."


In [21]:
from anthropic import AnthropicFoundry

endpoint = "https://510-ai-research.services.ai.azure.com/anthropic"
deployment_name = "claude-sonnet-5"

client = AnthropicFoundry(
    api_key=subscription_key,
    base_url=endpoint
)

results = []
for _, row in data.iterrows():
    entry = {"user_input": row["user_input"], "bot_output": row["bot_output"]}
    
    for prompt_name, prompt_template in PROMPTS.items():
        completion = client.messages.create(
            model=deployment_name,
            max_tokens=1024,
            messages=[{"role": "user", 
                       "content": prompt_template.format(
                                  question=row["user_input"],
                                  answer=row["bot_output"]
            )}],
        )
        raw = next(block.text for block in completion.content if block.type == "text").strip()
        score = next((int(l.split(":")[1].strip()) for l in raw.splitlines() if l.startswith("SCORE:")), None)
        reasoning = next((l.split(":")[1].strip() for l in raw.splitlines() if l.startswith("REASONING:")), raw)
        
        entry[f"{prompt_name}_score"] = score
        entry[f"{prompt_name}_reasoning"] = reasoning
    
    results.append(entry)

claude_df = pd.DataFrame(results)

In [22]:
from openai import OpenAI

endpoint = "https://510-ai-research.services.ai.azure.com/openai/v1"
deployment_name = "grok-4.3"

client = OpenAI(
    base_url=endpoint,
    api_key=subscription_key
)

results = []
for _, row in data.iterrows():
    entry = {"user_input": row["user_input"], "bot_output": row["bot_output"]}
    
    for prompt_name, prompt_template in PROMPTS.items():
        completion = client.chat.completions.create(
            model=deployment_name,
            messages=[{"role": "user", 
                       "content": prompt_template.format(
                                  question=row["user_input"],
                                  answer=row["bot_output"]
            )}],
        )
        raw = completion.choices[0].message.content.strip()
        score = next((int(l.split(":")[1].strip()) for l in raw.splitlines() if l.startswith("SCORE:")), None)
        reasoning = next((l.split(":")[1].strip() for l in raw.splitlines() if l.startswith("REASONING:")), raw)
        
        entry[f"{prompt_name}_score"] = score
        entry[f"{prompt_name}_reasoning"] = reasoning
    
    results.append(entry)

grok_df = pd.DataFrame(results)

In [23]:
from openai import OpenAI

endpoint = "https://510-ai-research.services.ai.azure.com/openai/v1"
deployment_name = "gpt-5.4"

client = OpenAI(
    base_url=endpoint,
    api_key=subscription_key
)

results = []
for _, row in data.iterrows():
    entry = {"user_input": row["user_input"], "bot_output": row["bot_output"]}
    
    for prompt_name, prompt_template in PROMPTS.items():
        completion = client.chat.completions.create(
            model=deployment_name,
            messages=[{"role": "user", 
                       "content": prompt_template.format(
                                  question=row["user_input"],
                                  answer=row["bot_output"]
            )}],
        )
        raw = completion.choices[0].message.content.strip()
        score = next((int(l.split(":")[1].strip()) for l in raw.splitlines() if l.startswith("SCORE:")), None)
        reasoning = next((l.split(":")[1].strip() for l in raw.splitlines() if l.startswith("REASONING:")), raw)
        
        entry[f"{prompt_name}_score"] = score
        entry[f"{prompt_name}_reasoning"] = reasoning
    
    results.append(entry)

gpt_df = pd.DataFrame(results)

In [25]:
final_df = pd.concat([deepseek_df, claude_df, grok_df, gpt_df], axis=0, ignore_index=True)
final_df.to_csv('../data/llm_validation_results.csv', index=False)